In [ ]:
import os
import numpy as np
import pandas as pd
import logging
import optuna

from sklearn.metrics import (accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, make_scorer)
from imblearn.metrics import specificity_score
from sklearn.metrics import roc_curve
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cross_decomposition import PLSRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from mylibs.confoundregressor import remove_confounds_linear1
from sklearn.inspection import permutation_importance
import shap

# --------------------------------------------------
# Logging
# --------------------------------------------------
logging.basicConfig(filename="nested_cv_log.txt", level=logging.INFO)


os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"


# --------------------------------------------------
# Load
# --------------------------------------------------
os.chdir('/scratch/project_XX/AnalysisFiles/Data_for_conventional_ML_analysis')

spectral_data = np.load('preprocessed_spectral_data_averaged_new.npy', allow_pickle=True)
class_labels = np.load('class_labels_averaged_new.npy', allow_pickle=True)
wavenumbers = np.load('wavenumbers.npy', allow_pickle=True)


sample_ids = spectral_data[:, 0]
XX = spectral_data[:, 1:].astype(np.float32)
YY = class_labels.astype(np.int32)

df_spectral_data = pd.DataFrame({
    "sample_id": sample_ids
})

logging.info(f"Data shape: {XX.shape}")
# --------------------------------------------------
# Load clinical data
# --------------------------------------------------
os.chdir('/scratch/project_2015104/AnalysisFiles/Data_matrices_averaged_samples')
df_clinical_data = pd.read_csv('cleaned_clinical_data.csv')
# Construct confound matrix
sex = pd.Categorical(df_clinical_data['GENDER']).codes
age = df_clinical_data['AGE_AT_SAMPLING'].values
C = np.column_stack((age, sex))


# --------------------------------------------------------
# Load clinical data and select RA seropositive and negative indices
# --------------------------------------------------------

# convert sample IDs to strings
#sample_ids = spectral_data[:, 0].astype(str)

# ICD10 filtering
# RA_pos = df_clinical_data[df_clinical_data['ICD10'].str.startswith('M05')]
# RA_neg = df_clinical_data[df_clinical_data['ICD10'].str.startswith('M06')]

# # convert sample IDs to strings
# RA_pos_ids = RA_pos['sample_ID'].astype(str).values
# RA_neg_ids = RA_neg['sample_ID'].astype(str).values

# # masks
# RA_pos_idx = np.isin(sample_ids, RA_pos_ids)
# RA_neg_idx = np.isin(sample_ids, RA_neg_ids)

# # spectra
# spectra_RA_pos = spectral_data[RA_pos_idx][:, 1:].astype(np.float32)
# spectra_RA_neg = spectral_data[RA_neg_idx][:, 1:].astype(np.float32)

# # combine
# RA_spectra = np.vstack((spectra_RA_pos, spectra_RA_neg))
# XX = RA_spectra
# logging.info(f"Data shape: {XX.shape}")

# # labels
# N_RA_pos = spectra_RA_pos.shape[0]
# N_RA_neg = spectra_RA_neg.shape[0]

# YY = np.zeros(N_RA_pos + N_RA_neg, dtype=int)
# YY[N_RA_pos:] = 1   # 0 = RA+, 1 = RA-

# # confound data in the same order as the spectra
# df_RA = pd.concat([RA_pos, RA_neg], ignore_index=True)
# sex = pd.Categorical(df_RA['GENDER']).codes
# age = df_RA['AGE_AT_SAMPLING'].values
# C = np.column_stack((age, sex))


# --------------------------------------------------
# Models
# --------------------------------------------------
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", importance_type="gain", random_state=42),
    "KNN": KNeighborsClassifier(),
    "CART": DecisionTreeClassifier(random_state=42),
    "Naive Bayes": GaussianNB(),
}

# --------------------------------------------------
# Optuna objective-function
# --------------------------------------------------
def objective(trial, model_name, X_train, y_train, C):

    base_model = models[model_name]
    model = clone(base_model)

# Hyperparameter search spaces
    if model_name == "Random Forest":
        model.set_params(
            n_estimators=trial.suggest_int("n_estimators", 100, 1000),
            max_depth=trial.suggest_int("max_depth", 2, 20),
            class_weight='balanced'
        )

    elif model_name == "SVM":
        model.set_params(
            C=trial.suggest_float("C", 1e-3, 1e3, log=True),
            gamma=trial.suggest_float("gamma", 1e-4, 1, log=True),
            kernel=trial.suggest_categorical("kernel", ["linear", "rbf"]),
            probability=True,
            class_weight='balanced'
        )

    elif model_name == "XGBoost":
        model.set_params(
            n_estimators=trial.suggest_int("n_estimators", 10, 200),
            max_depth=trial.suggest_int("max_depth", 2, 10),
            learning_rate=trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True),
            scale_pos_weight=trial.suggest_float("scale_pos_weight", 0.5, 1.5),
            max_delta_step=trial.suggest_int("max_delta_step", 0, 10),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.3, 1.0)
        )

    elif model_name == "KNN":
        model.set_params(
            n_neighbors=trial.suggest_int("n_neighbors", 1, 30),
            weights='distance'
        )

    elif model_name == "CART":
        model.set_params(
            max_depth=trial.suggest_int("max_depth", 2, 20),
            min_samples_split=trial.suggest_int("min_samples_split", 2, 20),
            min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 20),
            class_weight='balanced'
        )

    elif model_name == "Naive Bayes":
        model.set_params(
            var_smoothing=trial.suggest_float("var_smoothing", 1e-12, 1e-6, log=True)
        )

    # Adjust training-data and validation-data based on the training data
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    roc_auc_scores = []
    for train_idx, val_idx in skf.split(X_train, y_train):

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]
        C_tr, C_val = C[train_idx], C[val_idx]

        # Adjust the training data and validation data for age and sex separately to avoid data leakage
        # Fit confound regression using TRAIN set only
        X_train_adj, _, beta_conf = remove_confounds_linear1(X_tr, C_tr)

        # Calculate the adjusted validation data using the beta values from the training set
        X_val_adj = X_val - (C_val @ beta_conf)

        # Train the model on the adjusted training data
        model.fit(X_train_adj, y_tr)

        # Validate the model on the adjusted validation data
        if hasattr(model, "predict_proba"):
            y_scores = model.predict_proba(X_val_adj)[:, 1]
        else:
            y_scores = model.decision_function(X_val_adj)

        roc_auc_scores.append(roc_auc_score(y_val, y_scores))


    return np.mean(roc_auc_scores)

# --------------------------------------------------
# Repeated Nested CV
# --------------------------------------------------
num_rep = 50
results = {name: [] for name in models.keys()}
XGBoost_shap_values = []
RF_shap_values = []
roc_data = {name: [] for name in models.keys()}

for rep in range(num_rep):

    print(f"\n====== REPETITION {rep+1} ======")

    outer_cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=242 + rep)

    # Outer CV-loop
    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(XX, YY)):

        print(f"\n--- OUTER FOLD {fold+1} ---")

        X_train, X_test = XX[train_idx], XX[test_idx]
        y_train, y_test = YY[train_idx], YY[test_idx]
        C_train, C_test = C[train_idx], C[test_idx]

        # Adjust the training data and testing data for age and sex separately to avoid data leakage
        # calculate the beta values for the confound regression using the training set
        X_train_adj, intercept, beta_conf = remove_confounds_linear1(X_train, C_train)
        # calculate the adjustment for the outer testing set using the beta values from the training set only
        confound_effect_test = C_test @ beta_conf     # (n_test, n_features)
        X_test_adj = X_test - confound_effect_test


        for model_name in models.keys():

            print(f"Tuning {model_name}...")

             
            study = optuna.create_study(direction="maximize")
            study.optimize(lambda trial: objective(trial, model_name, X_train, y_train, C_train), n_trials=30, n_jobs=15)

            best_model = clone(models[model_name])
            best_model.set_params(**study.best_params)

            if model_name == "SVM":
                best_model.set_params(probability=True, class_weight="balanced")
            elif model_name == "Random Forest":
                best_model.set_params(class_weight="balanced")
            elif model_name == "CART":
                best_model.set_params(class_weight="balanced")

            # train the best model on the adjusted training data
            best_model.fit(X_train_adj, y_train)

            # Test the model
            # Outer-loop predictions
            y_pred = best_model.predict(X_test_adj)

            # Score ROC/AUC for the analysis
            if hasattr(best_model, "predict_proba"):
                y_score = best_model.predict_proba(X_test_adj)[:, 1]
            elif hasattr(best_model, "decision_function"):
                y_score = best_model.decision_function(X_test_adj)
            else:
                y_score = y_pred

            # Save ROC data for each sample
            for sample_id, true_val, score_val in zip(sample_ids[test_idx], y_test, y_score):
                roc_data[model_name].append({
                    "repetition": rep + 1,
                    "fold": fold + 1,
                    "sample_id": sample_id,
                    "y_true": int(true_val),
                    "y_score": float(score_val)
                })

            res = {
                "repetition": rep + 1,
                "fold": fold + 1,
                "accuracy": accuracy_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred, average="binary"), # sama kuin sensitivity
                "specificity": specificity_score(y_test, y_pred, average="binary"),
                "precision": precision_score(y_test, y_pred, average="binary"),
                "f1": f1_score(y_test, y_pred, average="binary"),
                "roc_auc": roc_auc_score(y_test, y_score),
                "best_params": study.best_params
            }

            results[model_name].append(res)

             # SHAP for Random Forest
            if model_name == "Random Forest":
                explainer = shap.TreeExplainer(best_model)
                shap_values = explainer(X_test_adj)
                print("RF SHAP shape:", shap_values.values.shape)
                
                # Calculate the mean absolute SHAP values for the testing set
                mean_abs_shap = np.abs(shap_values.values[:, :, 1]).mean(axis=0)
                for feature_idx, abs_shap_val in enumerate(mean_abs_shap):
                    RF_shap_values.append({
                                         "repetition": rep + 1,
                                         "fold": fold + 1,
                                         "feature_index": feature_idx,
                                         "mean_shap_value": abs_shap_val})

            # SHAP values for XGBoost
            if model_name == "XGBoost":
                explainer = shap.TreeExplainer(best_model)
                shap_values = explainer(X_test_adj)

                # Calculate the mean absolute SHAP values for the testing set
                mean_abs_shap = np.abs(shap_values.values).mean(axis=0) 

                for feature_idx, abs_shap_val in enumerate(mean_abs_shap):
                    XGBoost_shap_values.append({
                                        "repetition": rep + 1,
                                        "fold": fold + 1,
                                        "feature_index": feature_idx,
                                        "importance": abs_shap_val})

            print(f"{model_name}: AUC={res['roc_auc']:.4f}, F1={res['f1']:.4f}")


# --------------------------------------------------
# Save the results 
# --------------------------------------------------
base_out_dir = "/results"
os.makedirs(base_out_dir, exist_ok=True)

for model_name, model_results in results.items():
    df = pd.DataFrame(model_results)
    out_dir = os.path.join(base_out_dir, model_name)
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, f"{model_name}_results.csv"), index=False)
    logging.info(f"Saved: {model_name}")

# Save the feature-importance results
RF_shap_values_df = pd.DataFrame(RF_shap_values)
os.makedirs(base_out_dir, exist_ok=True)
RF_shap_values_df.to_csv(os.path.join(base_out_dir, "RF_SHAP_values.csv"), index=False)

XGBoost_shap_values_df = pd.DataFrame(XGBoost_shap_values)
os.makedirs(base_out_dir, exist_ok=True)
XGBoost_shap_values_df.to_csv(os.path.join(base_out_dir, "XGBoost_SHAP_values.csv"), index=False)

for model_name, data in roc_data.items():
    df = pd.DataFrame(data)
    out_dir = os.path.join(base_out_dir, model_name)
    os.makedirs(out_dir, exist_ok=True)
    df.to_csv(os.path.join(out_dir, f"{model_name}_roc_data.csv"),index=False)
    logging.info(f"Saved ROC data: {model_name}")   